# NepError — Notebook 2: CHiPSAL-BERT Fine-Tuning
## Fine-tune the Nepali SOTA encoder on MNLI-Nepali, then extract failures

---

**Project:** NepError: A Hierarchical Diagnostic Taxonomy of Failure Modes in Nepali NLI  
**Author:** Pema Tshering Sherpa, RV University Bengaluru  
**Collaborator:** Prof. Bal Krishna Bal, KU ILP Lab, Kathmandu University  

---

### Background

**CHiPSAL-BERT** (`IRIISNEPAL/RoBERTa_Nepali_110M`) is a 110M-parameter RoBERTa model  
pre-trained on **27.5 GB of Nepali news text** from 99 Nepali websites. It is the strongest  
publicly available Nepali-specific language model (NER F1: 93.74, POS F1: 97.52).

The public checkpoint is a **base MLM model only** — it has never seen NLI labels.  
This notebook adds a 3-class classification head and fine-tunes it on MNLI-Nepali.

### What this notebook does

| Step | Description |
|------|-------------|
| 1 | Load MNLI-Nepali and create train/validation/test splits |
| 2 | Tokenise with CHiPSAL-BERT's own SentencePiece tokenizer |
| 3 | Fine-tune with AdamW + linear warmup schedule |
| 4 | Evaluate on test split, print full classification report |
| 5 | Save best checkpoint to Google Drive (optional) |
| 6 | Run inference on test split, extract failure cases |
| 7 | Save `results/failures_chipsalbert_mnli.csv` |

### Training dynamics for Dataset Cartography (Week 7)

This notebook also records **per-instance confidence at every epoch** on the training set.  
These dynamics are saved to `results/cartography_dynamics.csv` and used in Notebook 3  
to classify instances as easy / ambiguous / hard (Swayamdipta et al., EMNLP 2020).

### Expected runtime on Colab T4 GPU
- Tokenisation: ~3 minutes  
- Training (3 epochs, 40,800 rows): ~35–45 minutes  
- Inference on test split (10,200 rows): ~6 minutes  

### Label encoding (same as Notebook 1)
```
0 = entailment
1 = neutral  
2 = contradiction
```

---
## Section 0 — Setup

In [ ]:
!pip install -q transformers datasets torch pandas numpy scikit-learn tqdm accelerate

In [ ]:
import os, json, warnings, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, accuracy_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Reproducibility — fix all random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────

@dataclass
class FinetuneConfig:
    # Model
    model_id: str      = "IRIISNEPAL/RoBERTa_Nepali_110M"
    model_name: str    = "chipsalbert"
    num_labels: int    = 3

    # Data
    dataset_id: str    = "IRIIS-RESEARCH/MNLI-Nepali"
    val_fraction: float = 0.10   # carve 10% of train → validation
    max_length: int    = 128

    # Training hyperparameters
    # These follow the recommendations in Sun et al. (2019) for BERT fine-tuning.
    num_epochs: int    = 3
    batch_size: int    = 16
    learning_rate: float = 2e-5
    weight_decay: float  = 0.01
    warmup_ratio: float  = 0.06   # 6% of total steps for linear warmup

    # Output
    output_dir: str    = "results"
    checkpoint_dir: str = "checkpoints"
    save_to_drive: bool = False   # set True to mount Google Drive and save checkpoint
    drive_path: str    = "/content/drive/MyDrive/NepError/checkpoints"

cfg = FinetuneConfig()
Path(cfg.output_dir).mkdir(exist_ok=True)
Path(cfg.checkpoint_dir).mkdir(exist_ok=True)

LABEL_NAMES = ['entailment', 'neutral', 'contradiction']
print("Config ready.")

---
## Section 1 — Load & Split Data

In [ ]:
print(f"Loading {cfg.dataset_id} ...")
raw = load_dataset(cfg.dataset_id)

# The MNLI-Nepali dataset has only train and test splits.
# We carve a validation set from train (stratified by label).
train_full = raw['train'].to_pandas()
test_df    = raw['test'].to_pandas()

# Stratified train/val split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_full,
    test_size=cfg.val_fraction,
    stratify=train_full['label'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train   : {len(train_df):,} rows")
print(f"Val     : {len(val_df):,} rows")
print(f"Test    : {len(test_df):,} rows")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts().sort_index().rename({0:'entailment',1:'neutral',2:'contradiction'}))

---
## Section 2 — Tokenise

In [ ]:
print(f"Loading tokenizer: {cfg.model_id}")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)

# Print vocab size — CHiPSAL-BERT was trained on Nepali so its
# SentencePiece vocabulary should segment Devanagari at morpheme
# boundaries much better than XLM-R's shared multilingual vocab.
print(f"Vocabulary size: {tokenizer.vocab_size:,}")


class NepaliNLIDataset(TorchDataset):
    """
    PyTorch Dataset for Nepali NLI pairs.

    Tokenises premise-hypothesis pairs using the provided tokenizer.
    Stores the original row index so we can track training dynamics
    per instance across epochs for Dataset Cartography.
    """
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int,
                 has_labels: bool = True):
        self.premises    = df['premise'].tolist()
        self.hypotheses  = df['hypothesis'].tolist()
        self.labels      = df['label'].tolist() if has_labels else None
        self.indices     = list(range(len(df)))  # original row index
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.has_labels  = has_labels

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'row_index':      torch.tensor(self.indices[idx], dtype=torch.long),
        }
        if self.has_labels:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = NepaliNLIDataset(train_df, tokenizer, cfg.max_length)
val_dataset   = NepaliNLIDataset(val_df,   tokenizer, cfg.max_length)
test_dataset  = NepaliNLIDataset(test_df,  tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"\nDataLoaders ready.")
print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

---
## Section 3 — Load Model & Set Up Optimiser

In [ ]:
print(f"Loading model: {cfg.model_id}")
print(f"Adding {cfg.num_labels}-class classification head...")

model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_id,
    num_labels=cfg.num_labels,
    # Suppress the expected warning about classification head weights
    # being randomly initialised (they are — we're about to train them)
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")

In [ ]:
# AdamW with weight decay, following standard BERT fine-tuning practice.
# We do NOT apply weight decay to bias terms and LayerNorm parameters.
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        'weight_decay': cfg.weight_decay,
    },
    {
        'params': [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0,
    },
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=cfg.learning_rate)

# Linear warmup then linear decay schedule
total_steps   = len(train_loader) * cfg.num_epochs
warmup_steps  = int(total_steps * cfg.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}")

---
## Section 4 — Training Loop with Dataset Cartography

In [ ]:
def evaluate(model, loader) -> Dict:
    """Run evaluation and return accuracy, loss, and per-instance predictions."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask).logits
            loss   = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return {'accuracy': accuracy, 'loss': avg_loss,
            'preds': all_preds, 'labels': all_labels}


# ── Training dynamics storage (for Dataset Cartography) ──────
# For each training instance, we record the model's softmax probability
# on the correct label at the end of every epoch.
# Shape will be: {row_index: [conf_epoch1, conf_epoch2, conf_epoch3]}
cartography: Dict[int, List[float]] = {i: [] for i in range(len(train_df))}

# ── Training ─────────────────────────────────────────────────
best_val_accuracy = 0.0
best_checkpoint_path = f"{cfg.checkpoint_dir}/best_chipsalbert_mnli.pt"
history = []
criterion = nn.CrossEntropyLoss()

print("Starting training...\n")

for epoch in range(1, cfg.num_epochs + 1):
    model.train()
    running_loss, n_correct, n_total = 0.0, 0, 0

    # Per-epoch cartography: collect (row_index, correct_label_prob)
    epoch_row_indices, epoch_correct_probs = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.num_epochs} [train]")
    for batch in pbar:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        row_indices    = batch['row_index']  # keep on CPU

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping — important for stability on small fine-tuning runs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        # Track training accuracy
        preds = torch.argmax(logits, dim=-1)
        n_correct += (preds == labels).sum().item()
        n_total   += labels.size(0)
        running_loss += loss.item()

        # Collect cartography data: probability assigned to the true label
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            for i, row_idx in enumerate(row_indices.numpy()):
                true_label_prob = probs[i][labels[i].item()]
                epoch_row_indices.append(int(row_idx))
                epoch_correct_probs.append(float(true_label_prob))

        pbar.set_postfix({'loss': f"{loss.item():.4f}",
                          'acc': f"{n_correct/n_total:.4f}"})

    # Store epoch cartography data
    for row_idx, prob in zip(epoch_row_indices, epoch_correct_probs):
        cartography[row_idx].append(prob)

    train_loss = running_loss / len(train_loader)
    train_acc  = n_correct / n_total

    # Validation
    val_results = evaluate(model, val_loader)

    print(f"  Epoch {epoch} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_results['loss']:.4f} | Val Acc: {val_results['accuracy']:.4f}")

    history.append({
        'epoch': epoch,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_results['loss'], 'val_acc': val_results['accuracy'],
    })

    # Save best checkpoint
    if val_results['accuracy'] > best_val_accuracy:
        best_val_accuracy = val_results['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': best_val_accuracy,
            'config': cfg.__dict__,
        }, best_checkpoint_path)
        print(f"  ✓ New best checkpoint saved (val_acc={best_val_accuracy:.4f})")

print(f"\nTraining complete. Best val accuracy: {best_val_accuracy:.4f}")

In [ ]:
# ── Save Dataset Cartography dynamics ────────────────────────
# For each training instance:
#   mean_confidence  = average prob on correct label across epochs → "easiness"
#   variability      = std of prob across epochs → how unstable the model was

carto_rows = []
for row_idx, conf_per_epoch in cartography.items():
    carto_rows.append({
        'row_index':       row_idx,
        'premise':         train_df.loc[row_idx, 'premise'],
        'hypothesis':      train_df.loc[row_idx, 'hypothesis'],
        'true_label':      train_df.loc[row_idx, 'label'],
        'mean_confidence': np.mean(conf_per_epoch),
        'variability':     np.std(conf_per_epoch),
        **{f'conf_epoch_{i+1}': v for i, v in enumerate(conf_per_epoch)},
    })

carto_df = pd.DataFrame(carto_rows)
carto_path = f"{cfg.output_dir}/cartography_dynamics.csv"
carto_df.to_csv(carto_path, index=False, encoding='utf-8-sig')
print(f"Cartography dynamics saved → {carto_path}")
print(f"  Rows: {len(carto_df):,}")
print(carto_df[['mean_confidence', 'variability']].describe().round(4))

---
## Section 5 — Evaluate Best Checkpoint on Test Set

In [ ]:
# Load the best checkpoint
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} "
      f"(val_acc={checkpoint['val_accuracy']:.4f})")

# Full test set evaluation
test_results = evaluate(model, test_loader)

print(f"\n{'='*50}")
print(f"TEST SET RESULTS — CHiPSAL-BERT on MNLI-Nepali")
print(f"{'='*50}")
print(f"Accuracy : {test_results['accuracy']:.4f} ({test_results['accuracy']*100:.2f}%)")
print(f"Loss     : {test_results['loss']:.4f}")
print(f"\nClassification Report:")
print(classification_report(
    test_results['labels'], test_results['preds'],
    target_names=LABEL_NAMES, zero_division=0
))

---
## Section 6 — Extract CHiPSAL-BERT Failure Cases

In [ ]:
import re

# Build full results DataFrame from test set
test_df_results = test_df.copy()
test_df_results.insert(0, 'instance_id',
    [f"MNLI-Nepali_test_{i}" for i in range(len(test_df))])
test_df_results['true_label'] = test_results['labels']
test_df_results['pred_label'] = test_results['preds']
test_df_results['correct']    = (
    test_df_results['true_label'] == test_df_results['pred_label']
).astype(int)
test_df_results['model']   = 'chipsalbert'
test_df_results['dataset'] = 'mnli'

# ── Extract failures with automated pre-filter flags ─────────
NEGATION_MARKERS = [
    'छैन', 'छैनन्', 'दैन', 'दैनन्', 'होइन', 'होइनन्',
    'नाई', 'नगर्', 'नहुने', 'नभएको', 'नगरेको'
]

def is_predominantly_latin(text: str, threshold: float = 0.5) -> bool:
    letters = re.findall(r'[a-zA-Z\u0900-\u097F]', text)
    if not letters:
        return False
    return (sum(1 for c in letters if c.isascii()) / len(letters)) > threshold

def has_code_switching(text: str) -> bool:
    return bool(re.search(r'[\u0900-\u097F]', text)) and \
           bool(re.search(r'[a-zA-Z]{2,}', text))

def has_negation(text: str) -> bool:
    return any(m in text for m in NEGATION_MARKERS)

failures = test_df_results[test_df_results['correct'] == 0].copy()

# Stratified sample: equal failure cases per true label, up to 200 each
n_per_class = min(200, failures['true_label'].value_counts().min())
failures_sampled = (
    failures
    .groupby('true_label', group_keys=False)
    .apply(lambda g: g.sample(n=n_per_class, random_state=SEED))
    .reset_index(drop=True)
)

combined = failures_sampled['premise'] + ' ' + failures_sampled['hypothesis']
failures_sampled['flag_romanization']  = combined.apply(is_predominantly_latin)
failures_sampled['flag_codeswitching'] = combined.apply(has_code_switching)
failures_sampled['flag_negation']      = combined.apply(has_negation)

# Blank annotation columns
failures_sampled['taxonomy_tier']  = ''
failures_sampled['annotator_note'] = ''
failures_sampled['annotator_id']   = ''

out_path = f"{cfg.output_dir}/failures_chipsalbert_mnli.csv"
failures_sampled.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"Total test failures    : {len(failures):,}")
print(f"Sampled for annotation : {len(failures_sampled):,}")
print(f"Saved → {out_path}")
print(f"\nPre-filter flag summary:")
print(f"  flag_negation      : {failures_sampled['flag_negation'].sum()}")
print(f"  flag_codeswitching : {failures_sampled['flag_codeswitching'].sum()}")
print(f"  flag_romanization  : {failures_sampled['flag_romanization'].sum()}")

---
## Section 7 — (Optional) Save Checkpoint to Google Drive

In [ ]:
if cfg.save_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(cfg.drive_path).mkdir(parents=True, exist_ok=True)

    import shutil
    dest = f"{cfg.drive_path}/best_chipsalbert_mnli.pt"
    shutil.copy(best_checkpoint_path, dest)
    print(f"Checkpoint saved to Google Drive: {dest}")
else:
    print("Google Drive save is disabled.")
    print("Set cfg.save_to_drive = True to persist the checkpoint across Colab sessions.")
    print(f"\nCurrent checkpoint location: {best_checkpoint_path}")
    print("Download it manually from the Colab file browser if needed.")

---
## Next Steps

After running this notebook you should have:
- `results/failures_chipsalbert_mnli.csv` — annotation-ready failure cases for CHiPSAL-BERT
- `results/cartography_dynamics.csv` — per-instance training dynamics for Dataset Cartography
- `checkpoints/best_chipsalbert_mnli.pt` — best fine-tuned model checkpoint

**Combined with Notebook 1, your annotation corpus is now complete:**
```
results/failures_xlmr_mnli.csv        (~200 sampled failures)
results/failures_mdeberta_mnli.csv    (~200 sampled failures)
results/failures_chipsalbert_mnli.csv (~200 sampled failures)
```

**Proceed to:** Manual annotation using the taxonomy guidelines.  
Open each CSV in Google Sheets / Label Studio, fill in `taxonomy_tier` for each row.